### Import Libaries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision import models
import matplotlib.pyplot as plt
import os

### Define Parameters

In [2]:
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 5
NUM_CLASSES = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Load and Preprocess Data

In [4]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

100%|████████████████████████████████████████████████████████████████| 170498071/170498071 [00:33<00:00, 5014875.30it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Number of training samples: 50000
Number of test samples: 10000


### Define Freeze Model

In [5]:
freeze_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in freeze_model.parameters():
    param.requires_grad = False

num_ftrs = freeze_model.fc.in_features
freeze_model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

freeze_model = freeze_model.to(device)
print(freeze_model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:02<00:00, 20.0MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### Define Fine-Tune Model

In [6]:
ft_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

num_ftrs = ft_model.fc.in_features
ft_model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

ft_model = ft_model.to(device)
print(ft_model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### Define Loss-function and Optimizer

In [8]:
criterion = nn.CrossEntropyLoss()
freeze_optimizer = optim.Adam(freeze_model.fc.parameters(), lr=LEARNING_RATE) #Only optimize the new FC layer
ft_optimizer = optim.Adam(ft_model.parameters(), lr=LEARNING_RATE)

### Define Training and Evaluating Function

In [9]:
def train(model, train_loader, criterion, optimizer, device, num_epochs = 5):
    train_losses = []
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return train_losses

In [10]:
def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

### Train Frozen Model

In [11]:
print("Training frozen model...")
losses_frozen = train(freeze_model, train_loader, criterion, freeze_optimizer, device, num_epochs=NUM_EPOCHS)
acc_frozen = evaluate(freeze_model, test_loader, device)
print(f"Accuracy on frozen model: {acc_frozen:.2f}%")

Training frozen model...
Starting training...
Epoch 1/5, Loss: 0.8331
Epoch 2/5, Loss: 0.6167
Epoch 3/5, Loss: 0.5907
Epoch 4/5, Loss: 0.5764
Epoch 5/5, Loss: 0.5684
Training complete.
Accuracy on frozen model: 80.97%


### Train Fine-tune Model

In [12]:
print("Training fine-tune model...")
losses_frozen = train(ft_model, train_loader, criterion, ft_optimizer, device, num_epochs=NUM_EPOCHS)
acc_ft = evaluate(ft_model, test_loader, device)
print(f"Accuracy on fine-tune model: {acc_ft:.2f}%")

Training fine-tune model...
Starting training...
Epoch 1/5, Loss: 0.5534
Epoch 2/5, Loss: 0.3187
Epoch 3/5, Loss: 0.2218
Epoch 4/5, Loss: 0.1601
Epoch 5/5, Loss: 0.1286
Training complete.
Accuracy on fine-tune model: 89.84%
